In [7]:

from google.colab import userdata

key = userdata.get("OPENROUTER_API_KEY")
print("KEY FOUND:", key is not None)
print("KEY PREFIX:", repr(key[:12]) if key else None)


KEY FOUND: True
KEY PREFIX: 'sk-or-v1-5ec'


In [8]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY").strip()

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

response = model.invoke("Say hello in exactly three words.")
print("RAW RESPONSE:", response)
print("CONTENT:", repr(response.content))


RAW RESPONSE: content='Hello dear friend' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 272, 'prompt_tokens': 23, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 280, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free', 'system_fingerprint': None, 'id': 'gen-1773179537-xWk6Zkj1FUGvqJQCJBxC', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cd9bc-88a3-7f03-99db-f193c047ca5d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 272, 'total_tokens': 295, 'input_token_details':

In [9]:
print("hello test")


hello test


In [ ]:
import os
import requests
from tavily import TavilyClient

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def internet_search(query: str) -> str:
    """Search internet and return top results"""
    result = tavily_client.search(query=query, max_results=3)
    return str(result)

@tool
def fetch_url(url: str) -> str:
    """Fetch website content"""
    response = requests.get(url, timeout=10)
    return response.text[:10000]

AGENT_PROMPT = """
You are a research assistant.

Steps:
1. Search the internet.
2. Select the top 3 websites.
3. Read them using fetch_url.
4. Answer the user question based on those sources.
"""

agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=AGENT_PROMPT
)

result = agent.invoke({
    "messages": [
        HumanMessage("What is LangGraph?")
    ]
})

print(result["messages"][-1].content)
